## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
import sys

class BoardMagic:
    def __init__(self, size, first_num, second_num, state):
        self.size = size
        self.first_num = first_num
        self.second_num = second_num
        self.state = state
        self.where = [0] * size
        self.operations = []

        for idx, val in enumerate(self.state):
            self.where[val] = idx

        self.unit = (first_num - second_num + size) % size
        self.unit = self.unit & -self.unit
        if self.unit == 0:
            self.unit = size

    def apply_add_real(self, val):
        if val == 0:
            return
        self.operations.append((2, val))
        for idx in range(self.size):
            self.state[idx] = (self.state[idx] + val) % self.size
        for idx, val_now in enumerate(self.state):
            self.where[val_now] = idx

    def apply_xor_real(self, val):
        if val == 0:
            return
        self.operations.append((1, val))
        for idx in range(self.size):
            self.state[idx] ^= val
        for idx, val_now in enumerate(self.state):
            self.where[val_now] = idx

    def record_add(self, val):
        if val:
            self.operations.append((2, val))

    def record_xor(self, val):
        if val:
            self.operations.append((1, val))

    def locate_pair(self, start, end):
        gap = (end - start + self.size - self.unit + self.size) % self.size
        left_shift = 0
        right_shift = 0
        half = self.size // 2

        while half >= 2 * self.unit:
            if gap >= half:
                gap -= half
                right_shift += half // 2
            else:
                left_shift += half // 2
            half //= 2

        low_part = start & (self.unit - 1)
        left_shift += self.size // 2 + low_part
        right_shift += low_part
        return left_shift, right_shift

    def exchange_values(self, x, y):
        if x == y:
            return

        if (x // self.unit) % 2 == (y // self.unit) % 2:
            if (x // self.unit) % 2 == 0:
                helper = (x & (self.unit - 1)) + self.unit
            else:
                helper = x & (self.unit - 1)

            self.exchange_values(x, helper)
            self.exchange_values(y, helper)
            self.exchange_values(x, helper)
            return

        base_left, base_right = self.locate_pair(self.first_num, self.second_num)
        need_left, need_right = self.locate_pair(x, y)

        self.record_add((need_left - x + self.size) % self.size)
        self.record_xor(need_left ^ base_left)
        self.record_add((self.first_num - base_left + self.size) % self.size)
        self.operations.append((0, 0))
        self.record_add((base_left - self.first_num + self.size) % self.size)
        self.record_xor(need_left ^ base_left)
        self.record_add((x - need_left + self.size) % self.size)

    def build_low_sequence(self, values, mod):
        used = [False] * mod
        for item in values:
            if item < 0 or item >= mod or used[item]:
                return False, []
            used[item] = True

        if mod == 1:
            return True, []

        left_part = [0] * (mod // 2)
        right_part = [0] * (mod // 2)

        for idx in range(mod // 2):
            left_part[idx] = values[2 * idx] // 2
            right_part[idx] = values[2 * idx + 1] // 2

        ok_left, seq_left = self.build_low_sequence(left_part, mod // 2)
        ok_right, seq_right = self.build_low_sequence(right_part, mod // 2)

        if not ok_left or not ok_right:
            return False, []

        sequence = []

        if values[0] % 2:
            sequence.append(1 if mod == 2 else -1)

        mark_left = 0
        for command in seq_left:
            if command > 0:
                sequence.extend([-1, 1])
            else:
                sequence.append(command * 2)
                mark_left ^= (-command * 2)

        if mark_left:
            sequence.append(-mark_left)

        mark_right = 0
        for command in seq_right:
            if command > 0:
                sequence.extend([1, -1])
            else:
                sequence.append(command * 2)
                mark_right ^= (-command * 2)

        if (mark_right & (mod // 2)) != (mark_left & (mod // 2)):
            for _ in range(mod // 4):
                sequence.extend([-1, 1])

        if mark_left >= mod // 2:
            mark_left -= mod // 2
        if mark_right >= mod // 2:
            mark_right -= mod // 2

        if mark_left != mark_right:
            return False, []

        compact = []
        for command in sequence:
            if not compact:
                compact.append(command)
            elif command < 0 and compact[-1] < 0:
                merged = -((-compact[-1]) ^ (-command))
                compact[-1] = merged
                if compact[-1] == 0:
                    compact.pop()
            else:
                compact.append(command)

        return True, compact

    def solve(self):
        if self.unit > 1:
            residues = [self.state[idx] & (self.unit - 1) for idx in range(self.unit)]
            possible, low_ops = self.build_low_sequence(residues, self.unit)
            if not possible:
                return False

            for command in low_ops:
                if command > 0:
                    self.apply_add_real(command)
                else:
                    self.apply_xor_real(-command)

        for rem in range(self.unit):
            needed = []
            current = []
            for idx in range(rem, self.size, self.unit):
                needed.append(idx)
                current.append(self.state[idx])
            if sorted(current) != needed:
                return False

        for rem in range(self.unit):
            for idx in range(rem, self.size, self.unit):
                while self.state[idx] != idx:
                    val = self.state[idx]
                    self.exchange_values(idx, val)

                    p1 = self.where[idx]
                    p2 = self.where[val]
                    self.state[p1], self.state[p2] = self.state[p2], self.state[p1]
                    self.where[idx], self.where[val] = p2, p1

        for idx, val in enumerate(self.state):
            if idx != val:
                return False

        return True

    def output(self):
        result = [str(len(self.operations))]
        for kind, value in self.operations:
            if kind == 0:
                result.append("0")
            else:
                result.append(f"{kind} {value}")
        print("\n".join(result))


def main():
    data = sys.stdin.read().split()
    if not data:
        return

    size = int(data[0])
    first_num = int(data[1])
    second_num = int(data[2])
    initial_state = list(map(int, data[3:3 + size]))

    solver = BoardMagic(size, first_num, second_num, initial_state)

    if solver.solve():
        solver.output()
    else:
        print("-1")


if __name__ == "__main__":
    main()

## B 长跑

In [ ]:
import sys
from bisect import bisect_left

def solve():
    # 使用迭代器读取所有输入，速度最快
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    
    it = iter(input_data)
    results = []
    
    while True:
        try:
            # 读取基础参数
            line = next(it)
            N = int(line)
            L = int(next(it))
            Maxn = int(next(it))
            S = int(next(it))
        except StopIteration:
            break
            
        stations = []
        for _ in range(N):
            p = int(next(it))
            c = int(next(it))
            stations.append((p, c))
        
        # 1. 基础情况判断
        if Maxn >= L:
            results.append("Yes")
            continue
        if N == 0:
            results.append("No")
            continue
            
        # 2. 预处理：排序补给站
        stations.sort()
        positions = [s[0] for s in stations]
        
        # 3. 动态规划
        # dp[i] 表示到达第 i 个补给站并补满体力所需的最少硬币
        # 因为硬币上限是 S，设一个比 S 大的值作为无穷大即可
        inf = S + 1
        dp = [inf] * N
        
        for i in range(N):
            pi, ci = stations[i]
            
            # 情况 A: 从起点直接到达
            if pi <= Maxn:
                dp[i] = ci
            
            # 情况 B: 从之前的补给站到达
            # 使用二分查找找到距离当前位置在 Maxn 范围内的第一个补给站索引
            start_idx = bisect_left(positions, pi - Maxn, 0, i)
            
            if start_idx < i:
                # 使用 Python 高效的内置 min 函数获取范围内的最小值
                prev_min = min(dp[start_idx:i])
                if prev_min + ci < dp[i]:
                    dp[i] = prev_min + ci
        
        # 4. 判断是否能有一条路径到达终点
        possible = False
        for i in range(N):
            # 如果到达该站的成本 <= S，且从该站能一口气跑到终点
            if dp[i] <= S and (L - stations[i][0]) <= Maxn:
                possible = True
                break
        
        results.append("Yes" if possible else "No")
    
    # 批量输出结果
    sys.stdout.write("\n".join(results) + "\n")

if __name__ == "__main__":
    solve()

## C 最长回文

In [ ]:
include <bits/stdc++.h>
using namespace std;

using ll = long long;

const ll MOD1 = 1000000007;
const ll MOD2 = 1000000009;
const ll BASE = 911382323;

struct H {
    ll x, y;
    bool operator==(const H& other) const {
        return x == other.x && y == other.y;
    }
};

struct Hash {
    vector<H> pre, pw;

    Hash() {}

    Hash(const string& s) {
        int n = s.size();
        pre.assign(n + 1, {0, 0});
        pw.assign(n + 1, {1, 1});

        for (int i = 0; i < n; i++) {
            pw[i + 1] = {
                pw[i].x * BASE % MOD1,
                pw[i].y * BASE % MOD2
            };

            pre[i + 1] = {
                (pre[i].x * BASE + s[i]) % MOD1,
                (pre[i].y * BASE + s[i]) % MOD2
            };
        }
    }

    H get(int l, int r) {
        if (l > r) return {0, 0};

        int len = r - l + 1;

        return {
            (pre[r + 1].x - pre[l].x * pw[len].x % MOD1 + MOD1) % MOD1,
            (pre[r + 1].y - pre[l].y * pw[len].y % MOD2 + MOD2) % MOD2
        };
    }
};

string build(const string& s) {
    string t = "$#";
    for (char c : s) {
        t += c;
        t += '#';
    }
    return t;
}

vector<int> manacher(const string& s) {
    int n = s.size();
    vector<int> p(n);

    for (int i = 0, l = 0, r = -1; i < n; i++) {
        int k = i > r ? 1 : min(p[l + r - i], r - i + 1);

        while (i - k >= 0 && i + k < n && s[i - k] == s[i + k]) {
            k++;
        }

        p[i] = k;

        if (i + k - 1 > r) {
            l = i - k + 1;
            r = i + k - 1;
        }
    }

    return p;
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int n;
    string A, B;
    cin >> n >> A >> B;

    string a = build(A);
    string b = build(B);

    int m = a.size();

    vector<int> pa = manacher(a);
    vector<int> pb = manacher(b);

    int ans = 0;

    for (int x : pa) ans = max(ans, x - 1);
    for (int x : pb) ans = max(ans, x - 1);

    string rb = b;
    reverse(rb.begin(), rb.end());

    Hash ha(a), hrb(rb);

    auto get_rev_b = [&](int l, int r) -> H {
        return hrb.get(m - 1 - r, m - 1 - l);
    };

    for (int i = 2; i < m; i++) {
        int len = max(pa[i], pb[i - 2]);

        int sa = i - len + 1;
        int sb = i + len - 3;

        int low = 0;
        int high = min(sa, m - 1 - sb);

        while (low < high) {
            int mid = (low + high + 1) >> 1;

            H left = ha.get(sa - mid, sa - 1);
            H right = get_rev_b(sb + 1, sb + mid);

            if (left == right) {
                low = mid;
            } else {
                high = mid - 1;
            }
        }

        ans = max(ans, len + low - 1);
    }

    cout << ans << '\n';

    return 0;
}


## D 优惠券

In [ ]:
include <bits/stdc++.h>
using namespace std;

struct BIT {
    int n;
    vector<int> tree;

    BIT(int n = 0) {
        init(n);
    }

    void init(int n_) {
        n = n_;
        tree.assign(n + 1, 0);
    }

    void add(int x, int v) {
        for (; x <= n; x += x & -x) {
            tree[x] += v;
        }
    }

    int sum(int x) {
        int res = 0;
        for (; x > 0; x -= x & -x) {
            res += tree[x];
        }
        return res;
    }

    int total() {
        return sum(n);
    }

    int kth(int k) {
        int pos = 0;
        for (int step = 1 << 20; step; step >>= 1) {
            int nxt = pos + step;
            if (nxt <= n && tree[nxt] < k) {
                pos = nxt;
                k -= tree[nxt];
            }
        }
        return pos + 1;
    }

    bool erase_first_ge(int l) {
        int before = sum(l - 1);
        if (total() == before) {
            return false;
        }

        int pos = kth(before + 1);
        add(pos, -1);
        return true;
    }
};

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int m;

    while (cin >> m) {
        vector<int> state(100001, 0);
        BIT bit(m);

        int ans = -1;

        for (int i = 1; i <= m; i++) {
            string op;
            cin >> op;

            if (op == "I") {
                int x;
                cin >> x;

                if (ans != -1) continue;

                if (state[x] > 0) {
                    if (!bit.erase_first_ge(state[x])) {
                        ans = i;
                    }
                }

                state[x] = i;
            } else if (op == "O") {
                int x;
                cin >> x;

                if (ans != -1) continue;

                if (state[x] <= 0) {
                    if (!bit.erase_first_ge(-state[x])) {
                        ans = i;
                    }
                }

                state[x] = -i;
            } else {
                if (ans == -1) {
                    bit.add(i, 1);
                }
            }
        }

        cout << ans << '\n';
    }

    return 0;
}


## E 任意点

In [ ]:
import sys

def solve():
    # 读取输入
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    
    n = int(input_data[0])
    points = []
    idx = 1
    for _ in range(n):
        x = int(input_data[idx])
        y = int(input_data[idx+1])
        points.append((x, y))
        idx += 2
    
    # 建图：邻接表
    adj = [[] for _ in range(n)]
    for i in range(n):
        for j in range(i + 1, n):
            # 如果横坐标相同或纵坐标相同，连边
            if points[i][0] == points[j][0] or points[i][1] == points[j][1]:
                adj[i].append(j)
                adj[j].append(i)
    
    # DFS 计算连通分量数量
    visited = [False] * n
    num_components = 0
    
    def dfs(u):
        visited[u] = True
        for v in adj[u]:
            if not visited[v]:
                dfs(v)
    
    for i in range(n):
        if not visited[i]:
            num_components += 1
            dfs(i)
    
    # 最少需要添加的边数为：分量数 - 1
    print(num_components - 1)

if __name__ == "__main__":
    solve()

## F 通配符匹配

In [ ]:
include <iostream>
include <string>
include <vector>

using namespace std;

constexpr long long M1 = 1e9 + 7;
constexpr long long M2 = 1e9 + 9;
constexpr long long B = 313;

long long p1_pow[200005], p2_pow[200005];
long long h1_arr[200005], h2_arr[200005];

// 预处理基数的幂次，方便后续哈希对比
void precompute() {
    p1_pow[0] = 1; p2_pow[0] = 1;
    for (int i = 1; i <= 200000; i++) {
        p1_pow[i] = (p1_pow[i - 1] * B) % M1;
        p2_pow[i] = (p2_pow[i - 1] * B) % M2;
    }
}

void solve() {
    string P;
    if (!(cin >> P)) return;
    int n;
    cin >> n;
    
    // 压缩相邻多余的 '*'
    string norm_P = "";
    for (char c : P) {
        if (c == '*') {
            if (norm_P.empty() || norm_P.back() != '*') {
                norm_P += c;
            }
        } else {
            norm_P += c;
        }
    }
    P = norm_P;
    
    bool has_star = (P.find('*') != string::npos);
    vector<string> parts;
    
    // 按照 '*' 分割出所有确切的小段片段
    if (has_star) {
        string cur = "";
        for (char c : P) {
            if (c == '*') {
                parts.push_back(cur);
                cur = "";
            } else {
                cur += c;
            }
        }
        parts.push_back(cur);
    }
    
    // 处理所有文件询问
    for (int i = 0; i < n; i++) {
        string S;
        cin >> S;
        
        // 若没有 '*'，只能精确长度硬匹配
        if (!has_star) {
            if (S.length() != P.length()) {
                cout << "NO\n";
                continue;
            }
            bool ok = true;
            for (size_t k = 0; k < P.length(); k++) {
                if (P[k] != '?' && P[k] != S[k]) {
                    ok = false; break;
                }
            }
            cout << (ok ? "YES" : "NO") << "\n";
            continue;
        }
        
        // 算出非通配符与 `?` 合计必须具备的最小长度
        size_t min_len = 0;
        for (const string& p : parts) min_len += p.length();
        if (S.length() < min_len) {
            cout << "NO\n";
            continue;
        }
        
        // 分别匹配首尾两端
        bool prefix_ok = true;
        if (!parts[0].empty()) {
            for (size_t k = 0; k < parts[0].length(); k++) {
                if (parts[0][k] != '?' && parts[0][k] != S[k]) {
                    prefix_ok = false; break;
                }
            }
        }
        if (!prefix_ok) { cout << "NO\n"; continue; }
        
        bool suffix_ok = true;
        if (!parts.back().empty()) {
            size_t s_len = S.length();
            size_t p_len = parts.back().length();
            for (size_t k = 0; k < p_len; k++) {
                if (parts.back()[k] != '?' && parts.back()[k] != S[s_len - p_len + k]) {
                    suffix_ok = false; break;
                }
            }
        }
        if (!suffix_ok) { cout << "NO\n"; continue; }
        
        // 预处理建立源文件的字符串前缀哈希
        h1_arr[0] = 0; h2_arr[0] = 0;
        for (size_t k = 0; k < S.length(); k++) {
            h1_arr[k + 1] = (h1_arr[k] * B + S[k]) % M1;
            h2_arr[k + 1] = (h2_arr[k] * B + S[k]) % M2;
        }
        
        // 贪心匹配首尾之间的各段
        int left = parts[0].length();
        int right = S.length() - parts.back().length() - 1;
        bool all_found = true;
        
        for (size_t idx = 1; idx < parts.size() - 1; idx++) {
            const string& Pi = parts[idx];
            int L = Pi.length();
            
            long long P_h1 = 0, P_h2 = 0;
            int q_pos_arr[15];
            long long p1_q_arr[15], p2_q_arr[15];
            int q_sz = 0;
            
            // 预先建立本段自身哈希，且记录 '?' 相对位置
            for (int k = 0; k < L; k++) {
                int val = (Pi[k] == '?') ? 0 : Pi[k];
                P_h1 = (P_h1 * B + val) % M1;
                P_h2 = (P_h2 * B + val) % M2;
                if (Pi[k] == '?') {
                    q_pos_arr[q_sz] = k;
                    p1_q_arr[q_sz] = p1_pow[L - 1 - k];
                    p2_q_arr[q_sz] = p2_pow[L - 1 - k];
                    q_sz++;
                }
            }
            
            int match_idx = -1;
            for (int j = left; j + L - 1 <= right; j++) {
                // 计算 S[j...j+L-1] 区间内的哈希值
                long long v1 = (h1_arr[j + L] - (h1_arr[j] * p1_pow[L]) % M1 + M1) % M1;
                long long v2 = (h2_arr[j + L] - (h2_arr[j] * p2_pow[L]) % M2 + M2) % M2;
                
                // 去除掉实际字符串在这段 '?' 落点上贡献的哈希值特征 (视为将其强行归0)
                for (int k = 0; k < q_sz; k++) {
                    long long char_val = S[j + q_pos_arr[k]];
                    long long sub1 = (char_val * p1_q_arr[k]) % M1;
                    v1 = (v1 - sub1 + M1) % M1;
                    long long sub2 = (char_val * p2_q_arr[k]) % M2;
                    v2 = (v2 - sub2 + M2) % M2;
                }
                
                // 当双哈希安全比对完全相同时
                if (v1 == P_h1 && v2 == P_h2) {
                    bool ok = true;
                    // 再加上一层O(L)基础字符检验（抵御纯粹的偶然概率双哈希碰撞）
                    for (int k = 0; k < L; k++) {
                        if (Pi[k] != '?' && Pi[k] != S[j + k]) {
                            ok = false;
                            break;
                        }
                    }
                    if (ok) {
                        match_idx = j;
                        break;
                    }
                }
            }
            
            if (match_idx == -1) {
                all_found = false;
                break;
            }
            left = match_idx + L; // 后一段必须在前一段之后开始
        }
        
        if (all_found) {
            cout << "YES\n";
        } else {
            cout << "NO\n";
        }
    }
}

int main() {
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);
    precompute();
    solve();
    return 0;
}

## G 汉诺塔

In [ ]:
import sys

def solve():
    # 读取输入
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    
    n = int(input_data[0])
    priorities = input_data[1:]
    
    # 柱子映射：A->0, B->1, C->2
    name_to_idx = {'A': 0, 'B': 1, 'C': 2}
    
    # 确定 1 个盘子时的转移目标
    # target_1[p] 表示 1 个盘子从 p 出发会去哪
    target_1 = [None] * 3
    for move in priorities:
        u = name_to_idx[move[0]]
        v = name_to_idx[move[1]]
        if target_1[u] is None:
            target_1[u] = v
            
    # DP 数组
    # f[i][p] 存储步数，to[i][p] 存储终点柱子
    f = [[0] * 3 for _ in range(n + 1)]
    to = [[0] * 3 for _ in range(n + 1)]
    
    # 初始化 n = 1
    for p in range(3):
        f[1][p] = 1
        to[1][p] = target_1[p]
        
    # 递推计算 2 到 n
    for i in range(2, n + 1):
        for p in range(3):
            b = to[i-1][p]      # i-1 个盘子先去的柱子
            c = 3 - p - b       # 剩下的那个空柱子
            
            # 判断 i-1 个盘子从 b 出发去哪
            if to[i-1][b] == c:
                # 情况 1: i-1 个盘子顺路去了 c
                f[i][p] = f[i-1][p] + 1 + f[i-1][b]
                to[i][p] = c
            else:
                # 情况 2: i-1 个盘子回到了 p，需要折腾两下最大盘子
                f[i][p] = f[i-1][p] + 1 + f[i-1][b] + 1 + f[i-1][p]
                to[i][p] = b
                
    # 输出从 A 柱 (0) 开始移动 n 个盘子的总步数
    print(f[n][0])

if __name__ == "__main__":
    solve()

## H 马步距离

In [ ]:
import sys

def solve():
    # 读取输入
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    
    xp, yp, xs, ys = map(int, input_data)
    
    # 1. 计算相对距离的绝对值
    x = abs(xp - xs)
    y = abs(yp - ys)
    
    # 2. 保证 x 是较大的那个，便于统一公式
    if x < y:
        x, y = y, x
        
    # 3. 处理极少数特殊点（参考题目图示中的标号）
    # (1, 0) 需要 3 步：(0,0)->(2,1)->(0,2)->(1,0)
    if x == 1 and y == 0:
        print(3)
        return
    # (2, 2) 需要 4 步：(0,0)->(1,2)->(2,0)->(0,1)->(2,2)
    if x == 2 and y == 2:
        print(4)
        return
        
    # 4. 使用通用公式计算
    # math.ceil(a/b) 可以写作 (a + b - 1) // b
    ans = max((x + 1) // 2, (x + y + 2) // 3)
    
    # 5. 奇偶性调整
    # 每一跳都会改变坐标之和的奇偶性，因此步数的奇偶性必须与距离和的奇偶性相同
    if ans % 2 != (x + y) % 2:
        ans += 1
        
    print(ans)

if __name__ == "__main__":
    solve()

## I 直方图最大矩形

In [ ]:

from typing import List

class Solution:
    def largestRectangleArea(self, heights: List[int]) -> int:
        # 如果数组为空，直接返回0
        if not heights:
            return 0
        
        # 在高度数组末尾添加一个高度为 0 的“哨兵”，
        # 确保遍历结束时，栈中所有剩余的柱子都能被弹出并计算面积。
        heights.append(0)
        
        # 栈中存储柱子的下标，初始化放入 -1 作为左边界的辅助
        stack = [-1]
        max_area = 0
        
        for i in range(len(heights)):
            # 当当前柱子的高度小于栈顶柱子的高度时，说明栈顶柱子找到了右边界
            while len(stack) > 1 and heights[i] < heights[stack[-1]]:
                # 弹出栈顶高度作为矩形的高
                h = heights[stack.pop()]
                # 此时新的栈顶就是该高度矩形的左边界（不含）
                # 当前 i 是该高度矩形的右边界（不含）
                w = i - stack[-1] - 1
                max_area = max(max_area, h * w)
            
            # 将当前下标压入栈中，保持栈内高度对应下标是单调递增的
            stack.append(i)
            
        # 养成好习惯，将添加的哨兵弹出，恢复原数组
        heights.pop()
        
        return max_area

## J 消防局的设立

In [ ]:
include <iostream>
include <vector>
include <algorithm>
include <queue>

using namespace std;

const int MAXN = 1005;
const int INF = 1e9;

vector<int> adj[MAXN];    // 邻接表
int parent[MAXN];         // 记录父节点
int nodes_by_depth[MAXN]; // 按深度排序的节点
int depth[MAXN];          // 节点的深度
int dist[MAXN];           // 节点到最近消防局的距离
int n;

// BFS获取深度和父节点关系
void bfs_info(int root) {
    for (int i = 1; i <= n; i++) depth[i] = -1;
    queue<int> q;
    q.push(root);
    depth[root] = 0;
    parent[root] = root; // 根节点的父节点设为自己

    int idx = 0;
    while (!q.empty()) {
        int u = q.front();
        q.pop();
        nodes_by_depth[++idx] = u; // 记录BFS序（自然满足深度递增）
        for (int v : adj[u]) {
            if (depth[v] == -1) {
                depth[v] = depth[u] + 1;
                parent[v] = u;
                q.push(v);
            }
        }
    }
}

// 更新消防局覆盖范围（BFS寻找距离在2以内的点）
void update_coverage(int start_node) {
    dist[start_node] = 0;
    queue<pair<int, int>> q;
    q.push({start_node, 0});

    while (!q.empty()) {
        int u = q.front().first;
        int d = q.front().second;
        q.pop();

        if (d >= 2) continue;

        for (int v : adj[u]) {
            if (dist[v] > d + 1) {
                dist[v] = d + 1;
                q.push({v, d + 1});
            }
        }
    }
}

int main() {
    if (!(cin >> n)) return 0;

    // 根据输入建立树结构
    // 题目描述：第i个输入是节点i+1的父节点
    for (int i = 2; i <= n; i++) {
        int p;
        cin >> p;
        adj[p].push_back(i);
        adj[i].push_back(p);
    }

    // 1. 初始化
    bfs_info(1);
    for (int i = 1; i <= n; i++) dist[i] = INF;

    int ans = 0;

    // 2. 贪心处理：从深度最大的节点开始向上倒序遍历
    for (int i = n; i >= 1; i--) {
        int u = nodes_by_depth[i];

        // 如果当前节点还未被覆盖（到最近消防局距离 > 2）
        if (dist[u] > 2) {
            ans++;
            // 贪心策略：在它的爷爷节点建消防局
            int grandfather = parent[parent[u]];
            update_coverage(grandfather);
        }
    }

    cout << ans << endl;

    return 0;
}